### **FACT ORDERS**

**Data Reading**

In [0]:
df=spark.sql("select * from databricks_cata.silver.orders_silver")

In [0]:
df.display()

order_id,customer_id,product_id,order_date,quantity,total_amount,year
O00001,C00710,P0159,2023-03-22T00:00:00.000Z,3,2022.87,2023
O00002,C00954,P0036,2023-06-30T00:00:00.000Z,2,3560.74,2023
O00003,C01578,P0427,2023-11-06T00:00:00.000Z,3,5903.52,2023
O00004,C00962,P0332,2024-02-27T00:00:00.000Z,3,4107.99,2024
O00005,C00156,P0038,2024-10-13T00:00:00.000Z,5,5784.95,2024
O00006,C00521,P0174,2023-05-17T00:00:00.000Z,5,407.75,2023
O00007,C00982,P0352,2024-01-18T00:00:00.000Z,4,4907.64,2024
O00008,C00976,P0172,2023-01-10T00:00:00.000Z,4,7037.88,2023
O00009,C01001,P0238,2023-04-20T00:00:00.000Z,3,4076.97,2023
O00010,C00702,P0258,2023-07-07T00:00:00.000Z,4,5695.64,2023


In [0]:
%sql
select * from databricks_cata.gold.dimproducts;

product_id,product_name,category,brand,price,discount_price,__START_AT,__END_AT
P0177,Article Smile,Clothing,Nike,1241.26,1117.134,2026-08-08T06:00:23.720Z,null
P0492,A Forward,Home,Nike,121.07,108.963,2026-08-08T06:00:23.720Z,null
P0242,Radio Lot,Home,Samsung,412.59,371.33099999999996,2026-08-08T06:00:23.720Z,null
P0063,Necessary Bad,Home,Nike,1876.88,1689.1920000000002,2026-08-08T06:00:23.720Z,null
P0367,Human See,Clothing,Nike,776.79,699.111,2026-08-08T06:00:23.720Z,null
P0067,Unit Election,Beauty,Samsung,144.72,130.248,2026-08-08T06:00:23.720Z,null
P0389,Necessary Upon,Beauty,Sony,458.59,412.731,2026-08-08T06:00:23.720Z,null
P0072,Station Fund,Toys,Revlon,1198.32,1078.488,2026-08-08T06:00:23.720Z,null
P0446,Blood National,Sports,Puma,1016.83,915.147,2026-08-08T06:00:23.720Z,null
P0477,Upon Sometimes,Electronics,Apple,917.67,825.903,2026-08-08T06:00:23.720Z,null


In [0]:
df_dimcus=spark.sql("select DimCustomerKey,customer_id as dim_customer_id from databricks_cata.gold.dimcustomers")

df_dimpro=spark.sql("select product_id as DimProductKey,product_id as dim_product_id from databricks_cata.gold.dimproducts")

**Fact Dataframe**

In [0]:
df_fact=df.join(df_dimcus,df.customer_id==df_dimcus.dim_customer_id,"left").join(df_dimpro,df.product_id==df_dimpro.DimProductKey,"left")
df_fact_new=df_fact.drop('dim_customer_id','dim_product_id','customer_id','product_id')

In [0]:
df_fact_new.display()

order_id,order_date,quantity,total_amount,year,DimCustomerKey,DimProductKey
O00001,2023-03-22T00:00:00.000Z,3,2022.87,2023,1679,P0159
O00002,2023-06-30T00:00:00.000Z,2,3560.74,2023,588,P0036
O00003,2023-11-06T00:00:00.000Z,3,5903.52,2023,992,P0427
O00004,2024-02-27T00:00:00.000Z,3,4107.99,2024,1807,P0332
O00005,2024-10-13T00:00:00.000Z,5,5784.95,2024,1760,P0038
O00006,2023-05-17T00:00:00.000Z,5,407.75,2023,681,P0174
O00007,2024-01-18T00:00:00.000Z,4,4907.64,2024,1318,P0352
O00008,2023-01-10T00:00:00.000Z,4,7037.88,2023,589,P0172
O00009,2023-04-20T00:00:00.000Z,3,4076.97,2023,1938,P0238
O00010,2023-07-07T00:00:00.000Z,4,5695.64,2023,310,P0258


**Upsert on Fact Table**

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("databricks_cata.gold.FactOrders"):
    dlt_obj=DeltaTable.forName(spark,"databricks_cata.gold.FactOrders")
    dlt_obj.alias("t").merge(df_fact_new.alias("s"),"t.DimCustomerKey=s.DimCustomerKey and t.DimProductKey=s.DimProductKey and t.order_id=s.order_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_fact_new.write.format("delta").option("path","abfss://gold@databricksetese2.dfs.core.windows.net/FactOrders").saveAsTable("databricks_cata.gold.FactOrders")

In [0]:
%sql
select * from databricks_cata.gold.FactOrders

order_id,order_date,quantity,total_amount,year,DimCustomerKey,DimProductKey
O00001,2023-03-22T00:00:00.000Z,3,2022.87,2023,1679,P0159
O00002,2023-06-30T00:00:00.000Z,2,3560.74,2023,588,P0036
O00003,2023-11-06T00:00:00.000Z,3,5903.52,2023,992,P0427
O00004,2024-02-27T00:00:00.000Z,3,4107.99,2024,1807,P0332
O00005,2024-10-13T00:00:00.000Z,5,5784.95,2024,1760,P0038
O00006,2023-05-17T00:00:00.000Z,5,407.75,2023,681,P0174
O00007,2024-01-18T00:00:00.000Z,4,4907.64,2024,1318,P0352
O00008,2023-01-10T00:00:00.000Z,4,7037.88,2023,589,P0172
O00009,2023-04-20T00:00:00.000Z,3,4076.97,2023,1938,P0238
O00010,2023-07-07T00:00:00.000Z,4,5695.64,2023,310,P0258
